# BNN + CVaR + Chance Constraint ile Üretim Planlama

Bu notebook, **Bayesçi Sinir Ağı (BNN)** ile talep belirsizliğini modelleyip üç karar yaklaşımını karşılaştırır:

1. Beklenen maliyeti minimize eden **Sample Average Approximation (SAA)**,
2. Kuyruk riskini minimize eden **CVaR** modeli,
3. Hizmet seviyesi sağlayan **chance-constrained** üretim kararı.

Akış:

`veri → Pyro BNN → posterior predictive → senaryolar → Pyomo/HiGHS → risk-duyarlı karar`


In [ ]:
# Gerekirse:
# %pip install torch pyro-ppl numpy pandas matplotlib pyomo highspy

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample
import pyomo.environ as pyo

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)
torch.set_default_dtype(torch.float32)


## 1. Talep verisini oluştur

Gerçek projede bu bölüm ERP/MES, pazar, fiyat, kampanya, hava durumu veya tedarik zinciri verileriyle değiştirilir.


In [ ]:
n = 260
day = np.arange(n)
promotion = np.random.binomial(1, 0.22, n)
temperature = 21 + 8*np.sin(2*np.pi*day/60) + np.random.normal(0, 2.0, n)
season = np.sin(2*np.pi*day/30)
trend = day/(n-1)

true_mean = 110 + 18*season + 24*promotion + 0.45*temperature + 12*trend + 9*promotion*season
noise_scale = 6 + 5*promotion + 2*np.abs(season)  # heteroskedastik veri
demand = true_mean + np.random.normal(0, noise_scale, n)

df = pd.DataFrame({
    "day": day,
    "promotion": promotion,
    "temperature": temperature,
    "season": season,
    "trend": trend,
    "demand": demand,
})
df.head()


In [ ]:
features = ["promotion", "temperature", "season", "trend"]
X_np = df[features].to_numpy(np.float32)
y_np = df["demand"].to_numpy(np.float32)

X_mean = X_np.mean(0, keepdims=True)
X_std = X_np.std(0, keepdims=True) + 1e-6
y_mean = float(y_np.mean())
y_std = float(y_np.std() + 1e-6)

X = torch.tensor((X_np - X_mean)/X_std)
y = torch.tensor((y_np - y_mean)/y_std)


## 2. Pyro ile BNN

Ağırlıklara önsel dağılım koyuyoruz ve `AutoDiagonalNormal` ile yaklaşık posterior öğreniyoruz.

\[
q_\phi(w)\approx p(w\mid\mathcal D)
\]

Bu öğretim örneğinde gözlem gürültüsü tek bir `sigma` ile modelleniyor. Gerçek uygulamada heteroskedastik likelihood ayrıca modellenebilir.


In [ ]:
class BayesianDemandNN(PyroModule):
    def __init__(self, in_features, hidden=18):
        super().__init__()
        self.hidden = PyroModule[nn.Linear](in_features, hidden)
        self.hidden.weight = PyroSample(
            dist.Normal(0, 1).expand([hidden, in_features]).to_event(2)
        )
        self.hidden.bias = PyroSample(
            dist.Normal(0, 1).expand([hidden]).to_event(1)
        )
        self.out = PyroModule[nn.Linear](hidden, 1)
        self.out.weight = PyroSample(
            dist.Normal(0, 1).expand([1, hidden]).to_event(2)
        )
        self.out.bias = PyroSample(
            dist.Normal(0, 1).expand([1]).to_event(1)
        )

    def forward(self, x, y=None):
        mean = self.out(torch.tanh(self.hidden(x))).squeeze(-1)
        sigma = pyro.sample("sigma", dist.LogNormal(-1.0, 0.35))
        with pyro.plate("data", x.shape[0]):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)
        return mean

pyro.clear_param_store()
model = BayesianDemandNN(X.shape[1])
guide = AutoDiagonalNormal(model)
svi = SVI(model, guide, pyro.optim.Adam({"lr": 0.015}), loss=Trace_ELBO())

losses = []
for step in range(2200):
    losses.append(svi.step(X, y)/len(y))

plt.plot(losses)
plt.xlabel("SVI adımı")
plt.ylabel("ELBO / gözlem")
plt.show()


## 3. Posterior predictive senaryolar

Gelecek dönem için kampanya açık, sıcaklık yüksek ve sezon etkisi pozitif bir durum varsayıyoruz.


In [ ]:
future = np.array([[1.0, 28.0, 0.80, 1.06]], dtype=np.float32)
future_X = torch.tensor((future - X_mean)/X_std)

predictive = Predictive(
    model,
    guide=guide,
    num_samples=2500,
    return_sites=("obs", "_RETURN"),
)
samples = predictive(future_X)

demand_samples = (
    samples["obs"].detach().cpu().numpy().reshape(-1) * y_std + y_mean
)
demand_samples = np.clip(demand_samples, 0, None)

pd.Series(demand_samples).describe(percentiles=[0.05, 0.50, 0.90, 0.95, 0.99])


In [ ]:
plt.hist(demand_samples, bins=45, density=True, alpha=0.7)
for q, label in [(0.50, "q50"), (0.95, "q95"), (0.99, "q99")]:
    plt.axvline(np.quantile(demand_samples, q), linestyle="--", label=label)
plt.xlabel("Talep")
plt.ylabel("Yoğunluk")
plt.legend()
plt.show()


## 4. Ortak maliyet yapısı

Tek dönem üretim miktarı \(x\) olsun.

- üretim maliyeti: \(c_p x\)
- fazla stok maliyeti: \(c_h(x-D_s)^+\)
- stokout / eksik karşılama maliyeti: \(c_u(D_s-x)^+\)

Senaryo maliyeti:

\[
L_s(x)=c_p x+c_h(x-D_s)^+ + c_u(D_s-x)^+
\]


In [ ]:
rng = np.random.default_rng(SEED)
S = 600
scenario = rng.choice(demand_samples, S, replace=False)

production_cost = 2.0
holding_cost = 1.0
shortage_cost = 8.0
capacity = 190.0


## 5. Model A — Beklenen maliyet (SAA)

Bu model risk-nötrdür ve senaryoların ortalama maliyetini minimize eder.


In [ ]:
def solve_expected_cost(scenario):
    S = len(scenario)
    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S-1)
    m.x = pyo.Var(bounds=(0, capacity))
    m.excess = pyo.Var(m.S, domain=pyo.NonNegativeReals)
    m.shortage = pyo.Var(m.S, domain=pyo.NonNegativeReals)

    D = {s: float(scenario[s]) for s in range(S)}
    m.c1 = pyo.Constraint(m.S, rule=lambda M, s: M.excess[s] >= M.x - D[s])
    m.c2 = pyo.Constraint(m.S, rule=lambda M, s: M.shortage[s] >= D[s] - M.x)

    m.obj = pyo.Objective(
        expr=production_cost*m.x + (1/S)*sum(
            holding_cost*m.excess[s] + shortage_cost*m.shortage[s]
            for s in m.S
        )
    )
    pyo.SolverFactory("appsi_highs").solve(m)
    return float(pyo.value(m.x)), float(pyo.value(m.obj))

x_saa, obj_saa = solve_expected_cost(scenario)
x_saa, obj_saa


## 6. Model B — CVaR optimizasyonu

\[
\operatorname{CVaR}_{\alpha}(L)
=
\eta+\frac{1}{(1-\alpha)S}\sum_s z_s
\]

\[
z_s\ge L_s(x)-\eta,\qquad z_s\ge0
\]

Burada \(\alpha=0.95\). Model en kötü %5'lik maliyet kuyruğuna odaklanır.


In [ ]:
def solve_cvar(scenario, alpha=0.95):
    S = len(scenario)
    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S-1)
    m.x = pyo.Var(bounds=(0, capacity))
    m.excess = pyo.Var(m.S, domain=pyo.NonNegativeReals)
    m.shortage = pyo.Var(m.S, domain=pyo.NonNegativeReals)
    m.eta = pyo.Var()
    m.z = pyo.Var(m.S, domain=pyo.NonNegativeReals)

    D = {s: float(scenario[s]) for s in range(S)}
    m.c1 = pyo.Constraint(m.S, rule=lambda M, s: M.excess[s] >= M.x - D[s])
    m.c2 = pyo.Constraint(m.S, rule=lambda M, s: M.shortage[s] >= D[s] - M.x)

    def loss_expr(M, s):
        return (
            production_cost*M.x
            + holding_cost*M.excess[s]
            + shortage_cost*M.shortage[s]
        )

    m.cvar_excess = pyo.Constraint(
        m.S,
        rule=lambda M, s: M.z[s] >= loss_expr(M, s) - M.eta,
    )
    m.obj = pyo.Objective(
        expr=m.eta + (1/((1-alpha)*S))*sum(m.z[s] for s in m.S)
    )

    pyo.SolverFactory("appsi_highs").solve(m)
    return float(pyo.value(m.x)), float(pyo.value(m.obj))

x_cvar, obj_cvar = solve_cvar(scenario, alpha=0.95)
x_cvar, obj_cvar


## 7. Model C — Chance constraint

Hizmet seviyesi:

\[
P(D\le x)\ge 1-\epsilon
\]

İmpirik senaryo yaklaşımında tek değişkenli bu örnek için yaklaşık olarak:

\[
x\ge Q_{1-\epsilon}(D)
\]

kullanabiliriz.

Bu yaklaşım stokout olasılığını doğrudan kontrol eder.


In [ ]:
epsilon = 0.05
required_service_quantity = float(np.quantile(scenario, 1-epsilon))

def solve_chance_proxy(scenario, required_quantity):
    S = len(scenario)
    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S-1)
    m.x = pyo.Var(bounds=(required_quantity, capacity))
    m.excess = pyo.Var(m.S, domain=pyo.NonNegativeReals)
    m.shortage = pyo.Var(m.S, domain=pyo.NonNegativeReals)

    D = {s: float(scenario[s]) for s in range(S)}
    m.c1 = pyo.Constraint(m.S, rule=lambda M, s: M.excess[s] >= M.x - D[s])
    m.c2 = pyo.Constraint(m.S, rule=lambda M, s: M.shortage[s] >= D[s] - M.x)

    m.obj = pyo.Objective(
        expr=production_cost*m.x + (1/S)*sum(
            holding_cost*m.excess[s] + shortage_cost*m.shortage[s]
            for s in m.S
        )
    )
    pyo.SolverFactory("appsi_highs").solve(m)
    return float(pyo.value(m.x)), float(pyo.value(m.obj))

x_chance, obj_chance = solve_chance_proxy(scenario, required_service_quantity)
x_chance, obj_chance


## 8. Out-of-sample karar kalitesi karşılaştırması

Aynı posterior predictive dağılımdan ayrı bir test örneklemi kullanıyoruz.


In [ ]:
test = rng.choice(demand_samples, 2000, replace=True)

def realized_cost(x, d):
    return (
        production_cost*x
        + holding_cost*max(x-d, 0)
        + shortage_cost*max(d-x, 0)
    )

def empirical_cvar(costs, alpha=0.95):
    var = np.quantile(costs, alpha)
    tail = costs[costs >= var]
    return float(tail.mean())

rows = []
for name, x in [
    ("SAA / beklenen maliyet", x_saa),
    ("CVaR95", x_cvar),
    ("Chance constraint %95", x_chance),
]:
    costs = np.array([realized_cost(x, d) for d in test])
    rows.append({
        "Yaklaşım": name,
        "Üretim": x,
        "Beklenen maliyet": costs.mean(),
        "CVaR95": empirical_cvar(costs, 0.95),
        "Stokout olasılığı": np.mean(test > x),
        "Hizmet seviyesi": np.mean(test <= x),
    })

comparison = pd.DataFrame(rows)
comparison


## 9. Nasıl yorumlanmalı?

Beklenen maliyet modeli ortalamaya odaklanır. CVaR modeli kötü maliyet senaryolarını bastırır. Chance constraint ise doğrudan bir hizmet seviyesi hedefler.

Aynı BNN posterioru farklı **risk tercihlerine** göre farklı kararlar üretebilir. Bu nedenle “iyi uncertainty model” ile “iyi decision model” aynı şey değildir.

Gerçek projelerde:
- posterior calibration,
- senaryo sayısı duyarlılığı,
- OOD / distribution shift,
- kapasite ve çok dönemli stok dinamikleri,
- tedarikçi / lead-time korelasyonları,
- çok ürünlü MILP
ayrıca ele alınmalıdır.
